In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import hashlib

# Set seed for reproducibility
np.random.seed(42)
random.seed(42)

# ==================== CONFIGURATION ====================
NUM_TRANSACTIONS = 5000
FRAUD_PERCENTAGE = 0.12  # 12% fraudulent transactions
START_DATE = datetime(2024, 1, 1)
END_DATE = datetime(2024, 12, 31)

# ==================== DATA LISTS ====================

# UPI IDs (Senders - mostly individuals)
senders = [
    f"{name}@{bank}" 
    for name in ['rahul.k', 'priya.s', 'amit.kumar', 'neha.g', 'vijay.m', 
                  'deepa.r', 'suresh.n', 'anjali.p', 'manish.t', 'poonam.d',
                  'rohit.sharma', 'kavita.j', 'vikram.s', 'sonal.g', 'arjun.k']
    for bank in ['okhdfcbank', 'okicici', 'oksbi', 'paytm', 'phonepe', 'amazonpay']
][:100]  # 100 unique sender UPI IDs

# UPI IDs (Receivers - merchants + individuals)
receivers = [
    f"{merchant}@{provider}"
    for merchant in ['bigbazaar', 'dominos', 'zomato', 'swiggy', 'amazon', 'flipkart',
                     'uber', 'ola', 'reliance', 'tata', 'ajio', 'myntra', 'pizza_hut',
                     'kfc', 'mcdonalds', 'starbucks', 'cafe_coffee_day', 'dmart',
                     'spencers', 'more_supermarket', 'medplus', 'apollo_pharmacy',
                     'petrol_bp', 'petrol_hp', 'petrol_iocl', 'electricity_board',
                     'water_dept', 'gas_company', 'school_fees', 'college_fees']
    for provider in ['paytm', 'phonepe', 'razorpay', 'billdesk', 'ccavenue']
][:200]  # 200 unique receiver UPI IDs

# Add some individual receivers (P2P transactions)
receivers.extend([f"{name}@{bank}" for name in ['ramesh.g', 'seema.r', 'mohit.a'] 
                  for bank in ['okhdfcbank', 'okicici']][:50])

# Merchant categories
categories = ['Retail', 'Food & Restaurant', 'E-commerce', 'Transportation', 
              'Utilities', 'Education', 'Healthcare', 'Entertainment', 
              'Travel', 'Fuel', 'Groceries', 'P2P Transfer']

# Locations
locations = ['Delhi', 'Mumbai', 'Bengaluru', 'Chennai', 'Kolkata', 'Pune', 
             'Hyderabad', 'Ahmedabad', 'Jaipur', 'Lucknow', 'Online', 'Noida']

# Fraud types
fraud_types = ['None', 'QR_Swap', 'UPI_ID_Spoofing', 'Amount_Manipulation', 
               'Phishing_Link', 'Fake_Merchant', 'Payment_Redirection']

# ==================== GENERATE DATA ====================

def generate_qr_hash(upi_id, amount, timestamp):
    """Generate unique QR code hash"""
    unique_string = f"{upi_id}_{amount}_{timestamp}"
    return hashlib.md5(unique_string.encode()).hexdigest()[:12]

def generate_transaction_date():
    """Generate random datetime between START_DATE and END_DATE"""
    time_between = END_DATE - START_DATE
    days_between = time_between.days
    random_days = random.randint(0, days_between)
    random_seconds = random.randint(0, 24*60*60)
    return START_DATE + timedelta(days=random_days, seconds=random_seconds)

def generate_risk_score(is_fraud, fraud_type):
    """Generate realistic risk scores"""
    if is_fraud == 0:
        return round(np.random.uniform(0, 30), 2)
    else:
        if fraud_type == 'QR_Swap':
            return round(np.random.uniform(75, 98), 2)
        elif fraud_type == 'UPI_ID_Spoofing':
            return round(np.random.uniform(80, 95), 2)
        elif fraud_type == 'Amount_Manipulation':
            return round(np.random.uniform(70, 90), 2)
        elif fraud_type == 'Phishing_Link':
            return round(np.random.uniform(85, 99), 2)
        elif fraud_type == 'Fake_Merchant':
            return round(np.random.uniform(65, 88), 2)
        else:
            return round(np.random.uniform(60, 85), 2)

# ==================== MAIN DATASET GENERATION ====================

data = []

print(f"🚀 Generating {NUM_TRANSACTIONS} synthetic UPI transactions...")
print(f"📊 Fraud percentage: {FRAUD_PERCENTAGE*100}%")

for i in range(NUM_TRANSACTIONS):
    # Basic transaction details
    txn_id = f"TXN_{START_DATE.strftime('%Y%m')}_{i:06d}"
    timestamp = generate_transaction_date()
    
    # Sender and receiver
    sender = random.choice(senders)
    receiver = random.choice(receivers)
    
    # Ensure sender != receiver
    while receiver in senders[:20] and sender == receiver:  # Avoid same person
        receiver = random.choice(receivers)
    
    # Amount (different distributions for fraud vs legit)
    amount = 0
    if random.random() < FRAUD_PERCENTAGE:  # Fraud transaction
        is_fraud = 1
        fraud_type = random.choice(fraud_types[1:])  # Exclude 'None'
        # Fraud amounts - often round numbers or unusual
        amount = round(random.choice([10, 50, 100, 500, 1000, 5000, 10000, 50000, 100000]) * 
                      random.uniform(0.9, 1.1), 2)
    else:  # Legitimate transaction
        is_fraud = 0
        fraud_type = 'None'
        # Normal amounts
        if random.random() < 0.6:  # Small transactions
            amount = round(random.uniform(10, 2000), 2)
        else:  # Large transactions
            amount = round(random.uniform(2000, 50000), 2)
    
    # QR Code details
    qr_hash = generate_qr_hash(receiver, amount, timestamp)
    qr_type = 'Dynamic' if random.random() < 0.3 else 'Static'
    
    # Merchant category
    if receiver in receivers[:150]:  # Most receivers are merchants
        category = random.choice(categories[:-1])  # Exclude P2P
    else:
        category = 'P2P Transfer'
    
    # Location
    location = random.choice(locations)
    
    # Device ID
    device_id = f"DEV_{random.randint(1000, 9999)}"
    
    # Risk score
    risk_score = generate_risk_score(is_fraud, fraud_type)
    
    # Append to data
    data.append([
        txn_id, timestamp.strftime('%Y-%m-%d %H:%M:%S'), 
        sender, receiver, amount,
        qr_hash, qr_type, category, location,
        device_id, is_fraud, fraud_type, risk_score
    ])

# ==================== CREATE DATAFRAME ====================

columns = [
    'transaction_id', 'timestamp', 'sender_upi', 'receiver_upi', 'amount',
    'qr_code_hash', 'qr_type', 'merchant_category', 'location',
    'device_id', 'is_fraud', 'fraud_type', 'risk_score'
]

df = pd.DataFrame(data, columns=columns)

# Sort by timestamp
df = df.sort_values('timestamp').reset_index(drop=True)

# ==================== ADD SOME REALISTIC PATTERNS ====================

# Pattern 1: Same QR code reused for fraud
fraud_qrs = df[df['is_fraud'] == 1]['qr_code_hash'].sample(5).values
for qr in fraud_qrs:
    # Mark some transactions with same QR as fraud
    idx = df[df['qr_code_hash'] == qr].index
    df.loc[idx, 'is_fraud'] = 1
    df.loc[idx, 'fraud_type'] = 'QR_Swap'

# Pattern 2: High amount transactions have more fraud
high_amount_idx = df[df['amount'] > 50000].sample(frac=0.3).index
df.loc[high_amount_idx, 'is_fraud'] = 1
df.loc[high_amount_idx, 'fraud_type'] = 'Amount_Manipulation'

# Pattern 3: Certain merchants are fake
fake_merchants = ['amazon', 'flipkart']  # Example - these are actually legitimate
# But for dataset, we mark some as fake
df.loc[df['receiver_upi'].str.contains('amazon|flipkart').sample(frac=0.1).index, 'is_fraud'] = 1
df.loc[df['receiver_upi'].str.contains('amazon|flipkart').sample(frac=0.1).index, 'fraud_type'] = 'Fake_Merchant'

# Recalculate risk scores
for idx, row in df.iterrows():
    df.loc[idx, 'risk_score'] = generate_risk_score(row['is_fraud'], row['fraud_type'])

# ==================== SAVE TO CSV ====================

# Save full dataset
df.to_csv('upi_transactions_fraud_dataset.csv', index=False)
print(f"\n✅ Dataset generated successfully!")
print(f"📁 File: upi_transactions_fraud_dataset.csv")
print(f"📊 Total transactions: {len(df)}")
print(f"💰 Total amount: ₹{df['amount'].sum():,.2f}")
print(f"🔴 Fraud transactions: {df['is_fraud'].sum()} ({df['is_fraud'].mean()*100:.1f}%)")
print(f"🟢 Legitimate transactions: {len(df) - df['is_fraud'].sum()}")
print(f"\n📈 Fraud by type:")
print(df['fraud_type'].value_counts())

# Save a sample for quick testing
df.head(100).to_csv('upi_transactions_sample.csv', index=False)
print(f"\n📁 Sample file: upi_transactions_sample.csv (100 transactions)")

# ==================== DATASET STATISTICS ====================

print("\n" + "="*50)
print("📊 DATASET STATISTICS")
print("="*50)
print(f"\n💰 Amount Statistics:")
print(df['amount'].describe())
print(f"\n🔴 Fraud Rate by Merchant Category:")
print(df.groupby('merchant_category')['is_fraud'].mean().sort_values(ascending=False))
print(f"\n📍 Top 5 Locations by Fraud:")
print(df[df['is_fraud']==1]['location'].value_counts().head())

🚀 Generating 5000 synthetic UPI transactions...
📊 Fraud percentage: 12.0%

✅ Dataset generated successfully!
📁 File: upi_transactions_fraud_dataset.csv
📊 Total transactions: 5000
💰 Total amount: ₹58,993,318.57
🔴 Fraud transactions: 1050 (21.0%)
🟢 Legitimate transactions: 3950

📈 Fraud by type:
fraud_type
None                   3941
Fake_Merchant           589
Amount_Manipulation     113
UPI_ID_Spoofing         100
Payment_Redirection      94
QR_Swap                  92
Phishing_Link            71
Name: count, dtype: int64

📁 Sample file: upi_transactions_sample.csv (100 transactions)

📊 DATASET STATISTICS

💰 Amount Statistics:
count      5000.000000
mean      11798.663714
std       17877.759206
min           9.040000
25%         787.987500
50%        1680.145000
75%       19318.402500
max      109522.030000
Name: amount, dtype: float64

🔴 Fraud Rate by Merchant Category:
merchant_category
Food & Restaurant    0.244656
P2P Transfer         0.241026
Retail               0.228571
Healthca

In [2]:
df = pd.read_csv('upi_transactions_fraud_dataset.csv')

In [4]:
df.sample(10)

,transaction_id,timestamp,sender_upi,receiver_upi,amount,qr_code_hash,qr_type,merchant_category,location,device_id,is_fraud,fraud_type,risk_score
3338,TXN_202401_001352,2024-08-31 12:19:43,neha.g@phonepe,mcdonalds@ccavenue,9671.00,718ad9cbabc9,Static,E-commerce,Hyderabad,DEV_2940,0,NaN,6.17
639,TXN_202401_004978,2024-02-19 00:01:24,kavita.j@okicici,dmart@razorpay,1283.92,e6a213c7220c,Static,Groceries,Jaipur,DEV_3936,0,NaN,28.80
1608,TXN_202401_002634,2024-04-30 21:45:41,vikram.s@paytm,school_fees@paytm,968.84,71dc76121c7a,Dynamic,Transportation,Lucknow,DEV_3525,1,NaN,73.24
2496,TXN_202401_002726,2024-07-04 08:29:46,suresh.n@paytm,petrol_bp@razorpay,723.23,9d783a6a0221,Dynamic,Healthcare,Ahmedabad,DEV_4523,0,NaN,20.93
2243,TXN_202401_002582,2024-06-14 03:24:50,priya.s@phonepe,petrol_iocl@razorpay,18512.96,6fbcc76818d8,Static,Groceries,Chennai,DEV_8434,0,NaN,12.45
2048,TXN_202401_004696,2024-05-31 14:32:42,arjun.k@okhdfcbank,amazon@paytm,45079.56,c0a6e81a155a,Dynamic,Healthcare,Ahmedabad,DEV_7606,0,NaN,23.95
40,TXN_202401_002559,2024-01-03 21:58:17,neha.g@amazonpay,kfc@razorpay,10685.46,ef2e5e9c1bfe,Static,Travel,Ahmedabad,DEV_6727,0,NaN,27.57
3609,TXN_202401_002050,2024-09-19 23:11:21,manish.t@amazonpay,dominos@paytm,275.15,a336039d32b6,Dynamic,E-commerce,Delhi,DEV_5292,0,NaN,26.88
2049,TXN_202401_003469,2024-05-31 14:39:20,amit.kumar@okicici,petrol_bp@billdesk,1388.77,08cc30d66a26,Static,Utilities,Mumbai,DEV_1376,1,NaN,79.67
2107,TXN_202401_001727,2024-06-04 19:28:57,amit.kumar@amazonpay,reliance@phonepe,1761.55,f59a992b9ee1,Static,Travel,Noida,DEV_4819,0,NaN,6.15
